In [1]:
# @title Set up the dependencies (need to restart session after running to refresh dl4bi)

try:
    # Retrieve ssh key for github access
    from google.colab import userdata

    key = userdata.get("github").replace("\r", "")

    !mkdir -p /root/.ssh
    !echo "$key" > /root/.ssh/id_ed25519
    !chmod 600 /root/.ssh/id_ed25519
    !ssh-keyscan github.com > /root/.ssh/known_hosts
    !chmod 644 /root/.ssh/known_hosts
    !ssh -T git@github.com

    # Install dl4bi
    !if cd dl4bi; then git pull; else git clone --single-branch --branch autoregressive-sampling ssh://git@github.com/MLGlobalHealth/dl4bi.git && pip install -e dl4bi; fi
    # Install latest jax, otherwise jax.experimental imports fail
    !pip install --no-cache-dir -U jax[cuda12]
except ModuleNotFoundError:
    print("Not running in colab, skipping setup")


Not running in colab, skipping setup


In [2]:
# @title Mount google drive

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)

    !if cd results; then echo "already mounted"; else ln -s /content/drive/MyDrive/results results; fi
except ModuleNotFoundError:
    print("Running locally")
    !if cd results; then echo "already mounted"; else ln -s "/Users/pgrynfelder/Library/CloudStorage/GoogleDrive-wadh6460@ox.ac.uk/My Drive/results" results; fi

Running locally
already mounted


In [3]:
# @title Imports (rerun from here after restarting session)

from datetime import datetime
from pathlib import Path
from typing import Optional

import jax
import jax.numpy as jnp
from jax import jit, random
from omegaconf import DictConfig, OmegaConf
from tqdm import tqdm

from dl4bi.meta_learning.autoregressive import (
    analytic_gp,
    build_gp_dataloader,
    closest_first,
    furthest_first,
    invert_permutation,
    sample_autoreg,
    sample_diagonal,
)
from dl4bi.meta_learning.train_utils import instantiate, load_ckpt

device = jax.devices()[0]
cpu = jax.devices("cpu")[0]

In [4]:
# @title Initial set-up: load model, create dataloader

model_dir = Path("results")

models = list(model_dir.glob("**/*.ckpt"))
model_path = models[0]


config: OmegaConf
state, config = load_ckpt(model_path)

/Volumes/Src/dl4bi/.venv/lib/python3.11/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1175: UserWarning: Couldn't find sharding info under RestoreArgs. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file instead of directly from RestoreArgs. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was not found in jax.local_devices().
ERROR:root:Device cuda:0 was n

In [5]:
@jit
def apply(
    s_ctx: jax.Array,
    f_ctx: jax.Array,
    s_test: jax.Array,
    valid_lens_ctx: Optional[jax.Array] = None,
):
    return state.apply_fn(
        {"params": state.params, **state.kwargs},
        s_ctx,
        f_ctx,
        s_test,
        valid_lens_ctx,
        training=False,
        # rngs={"extra": rng_extra}, # what happens if this is omitted? is this used test-time?
    )


dataloader = build_gp_dataloader(config.data, config.kernel)

In [6]:
# @title Get analytic solution, diagonal TNP-KR, and autoregressive samples.


def run(
    seed: int,
    job_id: Optional[str] = None,
    Np=32,  # number of paths to sample autoregressively, will be rounded up to be a multiple of batch size
    B=32,  # how to batch the autoregressive sampling. this parameter only affects runtime efficiency
    ls: float | None = None,
):
    rng = random.key(seed)

    if not job_id:
        job_id = str(datetime.now())
    results_dir = Path(f"results/{job_id}")
    results_dir.mkdir(parents=True)

    jnp.save(results_dir / "seed.npy", seed)

    rng, rng_dataloader = jax.random.split(rng)

    # TODO: don't overwrite the og config
    config.data.batch_size = 1
    if ls is not None:
        config.kernel.kwargs.ls.kwargs = DictConfig(
            {"dist": "fixed", "kwargs": {"value": ls}}
        )
    print("Seed:", seed)
    print("GP:", config.kernel)
    print("Data:", config.data)

    dataloader = build_gp_dataloader(config.data, config.kernel)(rng_dataloader)
    # note: unpacking the size-1 batch
    (
        [s_ctx],
        [f_ctx],
        [valid_lens_ctx],
        [s],
        [f],
        _valid_lens_test,  # unused
        [var],
        [ls],
        _period,  # unused
    ) = next(dataloader)
    print(f"var {var}, ls {ls}")
    jnp.save(results_dir / "var.npy", var)
    jnp.save(results_dir / "ls.npy", ls)

    f_ctx = f_ctx[:valid_lens_ctx]
    s_ctx = s_ctx[:valid_lens_ctx]

    # Excluding the context points from test set for numerical stability of later analysis.
    num_ctx = config.data.num_ctx.max
    s_test = s[num_ctx:]
    f_test = f[num_ctx:]

    # Save everything that can be needed for later analysis
    jnp.save(results_dir / "s_ctx.npy", s_ctx)
    jnp.save(results_dir / "f_ctx.npy", f_ctx)
    jnp.save(results_dir / "s_test.npy", s_test)
    jnp.save(results_dir / "f_test.npy", f_test)
    jnp.save(results_dir / "s.npy", s)
    jnp.save(results_dir / "f.npy", f)

    # Exact analytic solution (using data without noise)
    kernel = instantiate(config.kernel.kwargs.kernel)
    analytic_mu, analytic_cov = analytic_gp(
        s[:valid_lens_ctx],
        f[:valid_lens_ctx],
        s_test,
        kernel,
        var,
        ls,
    )
    jnp.save(results_dir / "analytic_mu", analytic_mu)
    jnp.save(results_dir / "analytic_cov", analytic_cov)

    # Analytic solution but with noise
    analytic_mu, analytic_cov = analytic_gp(
        s_ctx,
        f_ctx,
        s_test,
        kernel,
        var,
        ls,
    )
    jnp.save(results_dir / "noisy_analytic_mu", analytic_mu)
    jnp.save(results_dir / "noisy_analytic_cov", analytic_cov)

    # Regular (diagonal) TNP-KR
    [diagonal_mu], [diagonal_sd] = apply(
        s_ctx[None], f_ctx[None], s_test[None]
    )  # note need to expand dims
    diagonal_mu, diagonal_sd = diagonal_mu.squeeze(), diagonal_sd.squeeze()
    diagonal_var = diagonal_sd**2
    jnp.save(results_dir / "diagonal_mu", diagonal_mu)
    jnp.save(results_dir / "diagonal_var", diagonal_var)

    # Put arrays on gpu explicitly
    s_ctx = jax.device_put(s_ctx, device)
    f_ctx = jax.device_put(f_ctx, device)
    s_test = jax.device_put(s_test, device)

    # Autoregressive paths sampling
    if Np > 0:
        num_iters = (Np - 1) // B + 1
        for strategy in [
            # "preserve",
            "diagonal",
            "ltr",
            "furthest",
            "random",
            "closest",
        ]:
            print(f"Strategy: {strategy}")
            paths, densities = [], []

            # there was significant overhead from reordering within each batch, hence do that here
            match strategy:
                case "diagonal":
                    # perhaps it might make sense to try reordering the diagonal paths as well?
                    # the model should be permutation-invariant though
                    idx = idx_inv = ...
                case "preserve":
                    idx = idx_inv = ...
                case "ltr":
                    idx = jnp.argsort(s_test, axis=None)
                    idx_inv = invert_permutation(idx)
                case "closest":
                    idx = closest_first(s_ctx, s_test)
                    idx_inv = invert_permutation(idx)
                case "furthest":
                    idx = furthest_first(s_ctx, s_test)
                    idx_inv = invert_permutation(idx)
                case "random":
                    # handled inside sample_paths, so identity here
                    idx = idx_inv = ...

            s_test = s_test[idx]

            for i in tqdm(range(num_iters)):
                rng, rng_i = random.split(rng)
                if strategy == "diagonal":
                    path, log_density = sample_diagonal(
                        rng_i, apply, s_ctx, f_ctx, s_test, B
                    )
                else:
                    path, log_density = sample_autoreg(
                        rng_i,
                        apply,
                        s_ctx,
                        f_ctx,
                        s_test,
                        B,
                        random=(strategy == "random"),
                    )
                paths.append(path)
                densities.append(log_density)

            paths = jnp.concat(paths, axis=0)[:, idx_inv]
            jnp.save(results_dir / f"paths_{strategy}.npy", paths)
            densities = jnp.concat(densities, axis=0)
            jnp.save(results_dir / f"densities_{strategy}.npy", densities)

In [7]:
run(13, Np=1000, B=128)

Seed: 13
GP: {'cls': 'GP', 'kwargs': {'kernel': {'func': 'rbf'}, 'ls': {'cls': 'Prior', 'kwargs': {'dist': 'beta', 'kwargs': {'a': 3.0, 'b': 7.0}}}, 'var': {'cls': 'Prior', 'kwargs': {'dist': 'fixed', 'kwargs': {'value': 1.0}}}}}
Data: {'batch_size': 1, 'name': '1d', 'num_ctx': {'max': 50, 'min': 3}, 'obs_noise': 0.1, 's': [{'num': 100, 'start': -2, 'stop': 2}]}
var 1.0, ls 0.09566453844308853
Strategy: diagonal


100%|██████████| 8/8 [00:00<00:00, 14.13it/s]


Strategy: ltr


100%|██████████| 8/8 [00:56<00:00,  7.09s/it]


Strategy: furthest


100%|██████████| 8/8 [00:56<00:00,  7.05s/it]


Strategy: random


100%|██████████| 8/8 [01:09<00:00,  8.63s/it]


Strategy: closest


100%|██████████| 8/8 [00:57<00:00,  7.13s/it]
